In [42]:
import  torch
from torch import nn
from torch.utils.data import  DataLoader
from torchvision import  datasets
from torchvision.transforms import ToTensor

In [43]:
#download training data
training_data=datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)
#download the test data
test_data=datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

In [44]:
batch_size=64
#create data loaders
train_dataloader=DataLoader(training_data,batch_size=batch_size)
test_dataloader=DataLoader(test_data,batch_size=batch_size)
for X,y in test_dataloader:
    print(f"shape of X[N,C,H,W]: {X.shape}")
    print(f'Shape of y: {y.shape} {y.dtype}')
    break

shape of X[N,C,H,W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


Creating the model


In [45]:
#define the cpu
device="cpu" if torch.cuda.is_available() else "cpu"
class NN(nn.Module):
    def __init__(self):
        super(NN,self).__init__()
        self.flatten=nn.Flatten()
        self.linear_relu_stack=nn.Sequential(
            nn.Linear(28*28,512),
            nn.ReLU(),
            nn.Linear(512,512),
            nn.ReLU(),
            nn.Linear(512,10)
        )
    def forward(self,x):
        x=self.flatten(x)
        logits=self.linear_relu_stack(x)
        return  logits
model=NN().to(device)
print(model)

NN(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Optimizing the model parameters

In [46]:
loss_function=nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(model.parameters(),lr=1e-3)

In [47]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [48]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [49]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_function, optimizer)
    test(test_dataloader, model, loss_function)
print("Done!")

Epoch 1
-------------------------------
loss: 2.309875  [    0/60000]
loss: 2.296289  [ 6400/60000]
loss: 2.274876  [12800/60000]
loss: 2.269197  [19200/60000]
loss: 2.253516  [25600/60000]
loss: 2.228204  [32000/60000]
loss: 2.245541  [38400/60000]
loss: 2.210968  [44800/60000]
loss: 2.217512  [51200/60000]
loss: 2.180851  [57600/60000]
Test Error: 
 Accuracy: 28.6%, Avg loss: 2.175924 

Epoch 2
-------------------------------
loss: 2.192801  [    0/60000]
loss: 2.182031  [ 6400/60000]
loss: 2.127596  [12800/60000]
loss: 2.141353  [19200/60000]
loss: 2.100690  [25600/60000]
loss: 2.044988  [32000/60000]
loss: 2.085061  [38400/60000]
loss: 2.009752  [44800/60000]
loss: 2.027997  [51200/60000]
loss: 1.958643  [57600/60000]
Test Error: 
 Accuracy: 49.1%, Avg loss: 1.950502 

Epoch 3
-------------------------------
loss: 1.988173  [    0/60000]
loss: 1.959973  [ 6400/60000]
loss: 1.852948  [12800/60000]
loss: 1.890054  [19200/60000]
loss: 1.781903  [25600/60000]
loss: 1.734591  [32000/600

In [50]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


In [51]:
model=NN()
model.load_state_dict(torch.load("model.pth"))
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"
